<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Setting Up Vector Store
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

# Steps
- Set up
    - Import required libraries
    - Connect to Vantage Database
- Create a vector table
    - Create a table to hold text and id
    - Insert text data into the table
    - Create vector embeddings stored in a vector datatype
    - View the embeddings
- Using the vector table
    - TD_VectorDistance
    - Kmeans
    - HNSW

## Set up

### Import required libraries

In [1]:
# Teradata Vector Store libraries
from getpass import getpass
from teradatagenai import VSManager, VectorStore, VSPattern, VSApi
from teradataml import create_context, set_auth_token, execute_sql, display, DataFrame

### Connect to Vantage

In [2]:
# Connect to Vantage using create_context.
hostname = getpass(prompt = 'hostname: ')
username = getpass(prompt = 'username: ')
password = getpass(prompt = 'password: ')

context=create_context(host=hostname, username=username, password=password)

## Create a Vector Table

### Create a table to hold text and id

In [5]:
#Create a table for text to turn into embeddings later
execute_sql("""
CREATE TABLE text_data (
    id INTEGER GENERATED ALWAYS AS IDENTITY NOT NULL,
    text VARCHAR(1000),
    PRIMARY KEY (id)
);
""")


TeradataCursor uRowsHandle=119 bClosed=False

### Insert text data into the table

In [6]:
#Insert some text into the table
execute_sql("""
INSERT INTO text_data (text) VALUES
('The quick brown fox jumps over the lazy dog.');
INSERT INTO text_data (text) VALUES
('A journey of a thousand miles begins with a single step.');
INSERT INTO text_data (text) VALUES
('To be or not to be, that is the question.');
INSERT INTO text_data (text) VALUES
('All that glitters is not gold.');
INSERT INTO text_data (text) VALUES
('The only thing we have to fear is fear itself.');
INSERT INTO text_data (text) VALUES
('In the end, we will remember not the words of our enemies, but the silence of our friends.');
""")

TeradataCursor uRowsHandle=120 bClosed=False

In [7]:
#View table of text
df = DataFrame.from_query("SELECT * FROM text_data;")
df.head(10)

id,text
3,"To be or not to be, that is the question."
5,The only thing we have to fear is fear itself.
6,"In the end, we will remember not the words of our enemies, but the silence of our friends."
4,All that glitters is not gold.
2,A journey of a thousand miles begins with a single step.
1,The quick brown fox jumps over the lazy dog.


### Create Vector Embeddings stored in a Vector datatype

In [8]:
#Create embeddings for the text
execute_sql("""
CREATE TABLE text_embeddings AS (
    SELECT * FROM AI_TextEmbeddings(
        ON text_data AS InputTable
        USING
        authorization(AWSEmbeddingsAuth)
        region('us-west-2')
        apitype('aws')
        modelname('amazon.titan-embed-text-v1')
        textcolumn('text')
        outputformat('vector')
    ) as DT
) WITH DATA;   
""")

TeradataCursor uRowsHandle=137 bClosed=False

In [9]:
#View the embeddings
df_embeddings = DataFrame.from_query("SELECT * FROM text_embeddings;")
df_embeddings.head(10)

id,text,Embedding,Message
3,"To be or not to be, that is the question.","-0.163086,-0.675781,-0.0410156,-0.902344,0.462891,-0.273438,0.137695,0.000114441,-0.0415039,0.0461426,-0.238281,-0.855469,-0.261719,0.0756836,0.149414,0.0600586,-0.0703125,-0.890625,-0.294922,0.0615234,0.320312,-0.210938,-0.0913086,0.291016,-0.057373,0.234375,0.914062,-0.232422,0.257812,-0.00357056,0.396484,0.949219,0.279297,-0.310547,-0.00119781,-0.318359,0.746094,-0.123047,0.308594,-0.19043,-0.00369263,0.482422,0.235352,0.404297,0.355469,0.539062,-0.024292,-0.490234,0.691406,0.124023,0.189453,0.0593262,0.769531,-0.0361328,-0.205078,-0.10791,-0.427734,0.257812,-0.609375,0.0593262,-0.179688,0.0961914,-0.425781,-0.141602,-0.0192871,-0.174805,-0.486328,0.0150757,-0.0712891,-0.165039,-0.0115356,-0.230469,0.0145264,-0.0717773,0.0400391,0.310547,0.209961,0.21875,0.302734,0.0112305,-0.351562,-0.296875,-0.0319824,0.234375,-0.324219,0.375,0.15332,-0.347656,0.000137329,-0.170898,-0.300781,-0.0583496,0.253906,-0.566406,-0.0625,-0.244141,-0.130859,-0.0859375,0.636719,0.253906,-0.0913086,-0.103027,-0.246094,-0.0334473,0.",
5,The only thing we have to fear is fear itself.,"0.316406,-0.800781,-0.133789,-0.589844,0.902344,-0.597656,-0.182617,-2.52724e-05,0.394531,-0.0976562,-0.496094,0.357422,-0.261719,-0.212891,-0.194336,0.447266,-0.691406,-1.39062,-0.177734,0.265625,0.621094,-0.00221252,-0.310547,0.75,0.112793,-0.0529785,0.322266,-0.535156,-0.151367,-0.209961,0.396484,-0.376953,-0.179688,-0.515625,0.206055,-0.175781,0.144531,0.125977,-0.0947266,0.202148,0.139648,0.108398,0.628906,0.200195,-0.0527344,0.683594,-0.0771484,-0.0449219,0.472656,-0.000423431,0.0864258,-0.527344,0.523438,-0.0742188,-0.112793,0.224609,-0.0390625,-0.275391,-0.263672,-0.0610352,0.0437012,-0.511719,-0.265625,-0.241211,0.369141,-0.0218506,-0.451172,0.134766,-0.416016,-0.414062,-0.492188,-0.816406,-0.197266,-0.259766,0.664062,0.176758,-0.0488281,0.217773,0.273438,0.585938,-0.298828,-0.363281,0.139648,0.554688,0.0351562,0.112793,0.519531,-0.304688,0.000130653,0.285156,-0.242188,0.0493164,0.494141,-0.546875,0.032959,0.466797,-0.232422,-0.141602,0.0673828,0.482422,0.111328,-0.141602,-0.205078,-0.265625,0.119629",
6,"In the end, we will remember not the words of our enemies, but the silence of our friends.","0.176758,-0.296875,-0.410156,-0.0549316,-0.515625,0.347656,0.0524902,0.000120163,0.667969,0.163086,-0.386719,0.519531,-0.238281,0.0534668,-0.566406,-0.0529785,-0.0664062,-0.71875,0.11084,-0.220703,0.179688,-0.318359,0.0145874,0.462891,-0.679688,0.00354004,0.308594,-1.08594,0.0512695,-0.515625,0.417969,0.484375,0.287109,-0.703125,0.738281,-0.228516,0.800781,-0.182617,0.78125,-0.138672,-0.279297,-0.75,0.488281,-0.177734,-0.324219,0.166016,0.367188,0.0991211,0.248047,0.304688,0.103516,-0.355469,0.953125,0.273438,0.0996094,0.691406,0.367188,-0.236328,0.103027,-0.15918,0.337891,-0.462891,-0.707031,-0.257812,0.330078,-0.129883,0.345703,0.363281,0.117676,-0.00787354,-0.429688,-0.480469,-0.667969,0.574219,-0.0427246,0.150391,-0.255859,0.322266,0.249023,0.114746,-0.300781,0.253906,0.142578,-0.078125,-0.10498,0.582031,-0.0143433,-0.417969,0.000132561,0.597656,-0.347656,-0.194336,0.279297,-0.695312,-0.166016,0.373047,-0.220703,-0.0600586,0.421875,-0.22168,-0.11377,-0.226562,0.102051,-0.326172,0.349609,0.173828,0.613281,",
4,All that glitters is not gold.,"0.304688,-0.205078,0.057373,-0.675781,-0.144531,-0.367188,0.0116577,0.000377655,0.535156,0.112305,-0.155273,-0.570312,-0.419922,-0.367188,-0.251953,-0.333984,-0.111816,-1.33594,-0.265625,0.041748,0.375,-0.269531,-0.000679016,-0.0307617,0.191406,-0.310547,0.152344,-0.151367,0.494141,0.0235596,0.466797,0.118164,0.25,-0.6875,0.0898438,0.0415039,0.59375,0.304688,0.431641,0.376953,-0.294922,-0.0546875,0.100098,-0.00939941,0.378906,0.773438,-0.065918,0.131836,-0.644531,0.367188,-0.371094,0.339844,0.0874023,-0.0927734,0.172852,-1.03125,0.757812,-0.118164,-0.0615234,0.96875,0.120117,-0.0317383,-0.205078,-0.322266,-0.199219,0.519531,-0.302734,-0.188477,-0.445312,0.09

## Using the Vector Table

### TD_VectorDistance

In [10]:
#Create the query table and insert the query
execute_sql("""
CREATE TABLE query_table (
    id INTEGER GENERATED ALWAYS AS IDENTITY NOT NULL,
    query VARCHAR(1000),
    PRIMARY KEY (id)
);
""")
execute_sql("""
INSERT INTO query_table (query) VALUES
('What is the meaning of life?');
""")

#Create the query embeddings
execute_sql("""
CREATE TABLE query_embeddings AS (
    SELECT * FROM AI_TextEmbeddings(
        ON query_table AS InputTable
        USING
        authorization(AWSEmbeddingsAuth)
        region('us-west-2')
        apitype('aws')
        modelname('amazon.titan-embed-text-v1')
        textcolumn('query')
        outputformat('vector')
    ) as DT
) WITH DATA;   
""")


TeradataCursor uRowsHandle=156 bClosed=False

In [11]:
#View the query embeddings
df_query_embeddings = DataFrame.from_query("SELECT * FROM query_embeddings;")
df_query_embeddings.head(10)

id,query,Embedding,Message
1,What is the meaning of life?,"0.419922,-0.726562,-0.0800781,0.412109,-0.0566406,0.378906,-0.0595703,-0.001297,-0.375,0.296875,-0.107422,-0.785156,0.0219727,-0.248047,0.652344,0.214844,-0.0380859,-0.765625,0.0639648,0.507812,0.365234,-0.279297,-0.0563965,0.511719,-0.808594,0.0224609,1.14844,-0.738281,1.17188,-0.636719,-0.617188,0.988281,-0.378906,-1.17969,0.148438,0.0412598,0.542969,-0.867188,-0.0859375,0.0505371,0.722656,-0.0922852,0.398438,0.777344,-0.220703,1.09375,0.0432129,-0.691406,0.621094,0.090332,-0.00759888,-0.367188,1.77344,0.273438,0.451172,-0.196289,0.0922852,-0.0541992,0.249023,0.316406,0.195312,0.683594,-0.445312,-0.238281,0.164062,-0.71875,0.0339355,0.0834961,-0.0952148,0.277344,-0.0595703,-0.0510254,-0.0732422,-0.625,0.211914,0.435547,0.181641,0.597656,0.5,0.789062,0.431641,-0.166992,0.445312,0.652344,-0.380859,0.239258,-0.019165,-0.155273,0.000119686,0.144531,-0.441406,-0.166016,1.25,0.0339355,-0.621094,0.4375,-0.511719,-0.353516,-0.0568848,0.601562,-0.449219,0.235352,0.0198975,-0.414062,-0.324219,0.476562,0.435547,0.9882",


In [12]:
#Implement the TD_VectorDistance function to calculate the distance between two vectors
td_vector_distance_query = """
SELECT target_id, reference_id, distancetype, cast(distance as decimal(36,15)) as distance FROM TD_VECTORDISTANCE (
ON query_embeddings as TargetTable
ON text_embeddings as ReferenceTable Dimension
USING
TargetIDColumn('id')
TargetFeatureColumns('Embedding')
RefIDColumn('id')
RefFeatureColumns('Embedding')
DistanceMeasure('euclidean')
topk(5)
EmbeddingSize(1536)
) as dt;
"""

#Execute the query to get the distances
df_distances = DataFrame.from_query(td_vector_distance_query)
#Display the distances
df_distances.head(10)

target_id,reference_id,distancetype,distance
1,3,euclidean,19.083
1,4,euclidean,24.064
1,2,euclidean,23.210
1,6,euclidean,22.808
1,5,euclidean,22.584


In [13]:
# View the top result
top_result_query = """
SELECT text_data.text, text_data.id
FROM text_data
WHERE text_data.id = 3;
"""
# Execute the query to get the top result
top_result = DataFrame.from_query(top_result_query)
# Display the top result
top_result.head(10)

text,id
"To be or not to be, that is the question.",3


### TD_Kmeans

In [14]:
execute_sql("""
CREATE TABLE kmeans_centroids AS (
    SELECT * FROM TD_KMeans (
        ON text_embeddings AS InputTable
            USING
            IDColumn('id')
            TargetColumns('Embedding')
            NumClusters(2)
            Seed(1234)
            StopThreshold(0.0395)
            MaxIterNum(3)
            OutputClusterAssignment('true')
    ) AS DT) WITH DATA;
""")

# View the centroids
df_centroids = DataFrame.from_query("SELECT * FROM kmeans_centroids;")
# Display the centroids
df_centroids.head(10)

id,td_clusterid_kmeans
3,0
5,0
6,0
4,1
2,0
1,0


### TD_HNSW

In [15]:
#Create an index using HNSW
execute_sql("""
SELECT * FROM TD_HNSW(
    ON text_embeddings AS InputTable
    OUT PERMANENT TABLE ModelTable(text_embeddings_index)
    USING
    IDColumn('id')
    VectorColumn('Embedding')
    Seed(1234)
    EfConstruction(32)
    NumConnPerNode(32)
    MaxNumConnPerNode(32)
    DistanceMeasure('euclidean')
    EmbeddingSize(1536)
    ApplyHeuristics('true')
) AS DT;
""")

# Execute the HNSW index creation
hnsw_index_result = DataFrame.from_query("""SELECT * FROM text_embeddings_index;""")
# Display the HNSW index result
hnsw_index_result.head(10)

TD_HNSW_Nodes
b'...'
b'...'
b'...'
b'...'
b'...'
b'...'
b'...'
b'...'
b'...'
b'...'


### Clean up 

In [16]:
#Drop any tables as clean up
execute_sql("""DROP TABLE text_data;""")
execute_sql("""DROP TABLE text_embeddings;""")
execute_sql("""DROP TABLE query_table;""")
execute_sql("""DROP TABLE query_embeddings;""")
execute_sql("""DROP TABLE kmeans_centroids;""")
execute_sql("""DROP TABLE text_embeddings_index;""")

TeradataCursor uRowsHandle=244 bClosed=False